In [ ]:
"""Day 27 / Session 1 - Lab Assignment 2: Top-k and Top-p (Nucleus) Filtering - Truncating the Sampling Distribution."""
import numpy as np


def softmax(logits):
    z = np.asarray(logits, dtype=float)
    e = np.exp(z - np.max(z))
    return e / e.sum()


def top_k_mask(probs, k):
    probs = np.asarray(probs, dtype=float)
    k = min(max(int(k), 0), len(probs))                      # clamp k to the vocabulary size
    order = np.argsort(-probs, kind="stable")
    mask = np.zeros(len(probs), dtype=bool)
    mask[order[:k]] = True
    return mask


def top_k_filter(probs, k):
    probs = np.asarray(probs, dtype=float)
    kept = np.where(top_k_mask(probs, k), probs, 0.0)
    return kept / kept.sum()                                 # renormalise; ratios between kept tokens are unchanged


def nucleus_size(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(-probs, kind="stable")
    cumulative = np.cumsum(probs[order])
    size = int(np.searchsorted(cumulative, p_thresh - 1e-12, side="left")) + 1    # smallest prefix with cumsum >= p
    return int(min(max(size, 1), len(probs)))                                     # always keep at least one token


def top_p_filter(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(-probs, kind="stable")
    keep = order[:nucleus_size(probs, p_thresh)]
    kept = np.zeros_like(probs)
    kept[keep] = probs[keep]
    return kept / kept.sum()
